In [14]:
import matplotlib.pyplot as plt
import torch
import torchvision
from torch import nn
from torchvision import transforms, datasets
from torchvision.datasets import ImageFolder
from torchinfo import summary
from torch.utils.data import DataLoader, Subset
import os

In [3]:
if torch.cuda.is_available():
    device = "cuda" # Use NVIDIA GPU 
else:
    device = "cpu"
device

'cpu'

In [5]:
# Veriyi Colab/bilgisayarına indirmesi ve başlatması için:
# data = datasets.Imagenette(root="./data", split="train", download=True, size="320px")

In [15]:
data_dir = "./data/imagenette2-320"

train_dir = os.path.join(data_dir, "train")
val_dir = os.path.join(data_dir, "val")

train_dataset = ImageFolder(root=train_dir, transform=transforms.ToTensor())
val_dataset = ImageFolder(root=val_dir, transform=transforms.ToTensor())

In [16]:
len(train_dataset)

9469

In [18]:
len(val_dataset)

3925

In [26]:
img, label = train_dataset[3031]

In [27]:
label

3

In [28]:
img

tensor([[[1.0000, 1.0000, 1.0000,  ..., 0.9882, 0.9882, 0.9882],
         [1.0000, 1.0000, 1.0000,  ..., 0.9882, 0.9882, 0.9882],
         [1.0000, 1.0000, 1.0000,  ..., 0.9882, 0.9882, 0.9882],
         ...,
         [0.8863, 0.8824, 0.8784,  ..., 0.8902, 0.8941, 0.8980],
         [0.8824, 0.8784, 0.8745,  ..., 0.8863, 0.8902, 0.8941],
         [0.8824, 0.8784, 0.8745,  ..., 0.8824, 0.8863, 0.8902]],

        [[1.0000, 1.0000, 1.0000,  ..., 0.9882, 0.9882, 0.9882],
         [1.0000, 1.0000, 1.0000,  ..., 0.9882, 0.9882, 0.9882],
         [1.0000, 1.0000, 1.0000,  ..., 0.9882, 0.9882, 0.9882],
         ...,
         [0.8824, 0.8784, 0.8745,  ..., 0.8510, 0.8549, 0.8588],
         [0.8784, 0.8745, 0.8706,  ..., 0.8471, 0.8510, 0.8549],
         [0.8784, 0.8745, 0.8706,  ..., 0.8431, 0.8471, 0.8510]],

        [[1.0000, 1.0000, 1.0000,  ..., 0.9412, 0.9412, 0.9412],
         [1.0000, 1.0000, 1.0000,  ..., 0.9412, 0.9412, 0.9412],
         [1.0000, 1.0000, 1.0000,  ..., 0.9412, 0.9412, 0.

In [29]:
img.shape

torch.Size([3, 320, 836])

In [24]:
class Fire(nn.Module):
    def __init__(self, in_channels = 3, s1 = 16, e1 = 64, e3 = 64):
        super().__init__()
        self.squeeze = nn.Conv2d(
            in_channels = in_channels,
            out_channels = s1,
            kernel_size = 1,
            padding = 0
            )
        self.activation = nn.ReLU(inplace = True)

        self.expand1 = nn.Conv2d(in_channels = s1,
                                out_channels = e1,
                                kernel_size = 1,
                                padding = 0)
        
        self.expand3 = nn.Conv2d(in_channels = s1,
                                out_channels = e3,
                                kernel_size = 3,
                                padding = 1)
    def forward(self, x):
        squeeze_out = self.activation(self.squeeze(x))

        out_e1 = self.activation(self.expand1(squeeze_out))
        out_e3 = self.activation(self.expand3(squeeze_out))

        return torch.cat([out_e1, out_e3], dim = 1)

In [25]:
class SqueezeNet(nn.Module):
    def __init__(self, classes = 10):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels = 3, 
                                out_channels = 96,
                                kernel_size = 7, 
                                stride = 2, 
                                padding = 3)
        self.relu1 = nn.ReLU(inplace=True)
        self.maxpool1 = nn.MaxPool2d(3, stride = 2)
        
        self.fire2 = Fire(in_channels = 96, s1 = 16, e1 = 64, e3 = 64)
        self.fire3 = Fire(in_channels = 128, s1 = 16, e1 = 64, e3 = 64)
        self.fire4 = Fire(in_channels = 128, s1 = 32, e1 = 128, e3 = 128)
        self.maxpool4 = nn.MaxPool2d(3, stride = 2)

        self.fire5 = Fire(in_channels = 256, s1 = 32, e1 = 128, e3 = 128)
        self.fire6 = Fire(in_channels = 256, s1 = 48, e1 = 192, e3 = 192)
        self.fire7 = Fire(in_channels = 384, s1 = 48, e1 = 192, e3 = 192)
        self.fire8 = Fire(in_channels = 384, s1 = 64, e1 = 256, e3 = 256)
        self.maxpool8 = nn.MaxPool2d(3, stride = 2)

        self.fire9 = Fire(in_channels = 512, s1 = 64, e1 = 256, e3 = 256)

        self.dropout =nn.Dropout(p=0.5)
        
        self.conv10 = nn.Conv2d(in_channels = 512, 
                                out_channels = classes,
                                kernel_size = 1, 
                                stride = 1)
        self.relu10 = nn.ReLU(inplace = True)
        self.avgpool10 = nn.AdaptiveAvgPool2d((1, 1))

    def forward(self, x):
        first = self.maxpool1(self.relu1(self.conv1(x)))
        blok_1 = self.maxpool4(self.fire4(self.fire3(self.fire2(first))))
        blok_2 = self.maxpool8(self.fire8(self.fire7(self.fire6(self.fire5(blok_1)))))
        blok_3 = self.avgpool10(self.relu10(self.conv10(self.dropout(self.fire9(blok_2)))))  # batch,10,1,1
        return torch.flatten(blok_3, 1)  # (batch,10) 0. indekse dokunma flattena 1den başla

In [34]:
img = img.permute(1,2,0)
img.shape

torch.Size([320, 836, 3])

In [35]:
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(size = 224, scale=(0.8, 1)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [41]:
train_data = ImageFolder(root = "./data/imagenette2-320/train", transform = train_transforms)
val_data = ImageFolder(root = "./data/imagenette2-320/val", transform = val_transforms)

train_loader = DataLoader(train_data, batch_size = 32, shuffle = True)
val_loader = DataLoader(val_data, batch_size = 32, shuffle = False)

In [42]:
model = SqueezeNet()
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params = model.parameters(), lr = 0.005)

In [47]:
img, label = next(iter(train_loader))

out = model(img)

print(out[:5])
print(label[:5])

tensor([[2.9507e-02, 3.5920e-05, 4.6258e-02, 4.5001e-02, 2.2440e-02, 1.9237e-02,
         1.3580e-02, 3.1980e-02, 3.3636e-02, 8.3061e-05],
        [2.9653e-02, 1.8543e-04, 4.0593e-02, 4.7486e-02, 2.0205e-02, 1.9291e-02,
         1.3492e-02, 3.1389e-02, 3.3976e-02, 3.2595e-04],
        [2.9452e-02, 9.0593e-05, 4.3101e-02, 4.5084e-02, 2.2614e-02, 1.9327e-02,
         1.3853e-02, 2.9770e-02, 3.4407e-02, 6.9886e-04],
        [2.8691e-02, 9.5885e-05, 4.4657e-02, 4.6339e-02, 2.1371e-02, 1.9452e-02,
         1.4857e-02, 3.1758e-02, 3.2632e-02, 4.9450e-04],
        [2.9370e-02, 2.0821e-04, 4.3742e-02, 4.6116e-02, 2.6156e-02, 2.0029e-02,
         1.4711e-02, 3.3077e-02, 3.4036e-02, 5.5245e-04]],
       grad_fn=<SliceBackward0>)
tensor([6, 6, 9, 9, 0])
